# AttackAgentDetect: public-trace feasibility study

This notebook reruns the pinned public-data audit, task-held-out baselines, and same-task retrieval diagnostics. It prints aggregate counts and metrics only; it does not display raw commands, prompts, or tool arguments.

**Interpretation boundary:** the public benchmark labels human-study sessions and model sessions, not attacker identity or shared campaign. Human terminal keystrokes and model tool calls are collected differently. Six independent test tasks make every interval exploratory.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_MANIFEST = ROOT / "data/raw/cyber-task-horizons/download_manifest.json"
if not DATA_MANIFEST.exists():
    subprocess.run([sys.executable, "scripts/download_public_traces.py"], check=True)

print(f"Repository: {ROOT}")
print("Pinned dataset manifest present:", DATA_MANIFEST.exists())

Repository: /Users/mac/Documents/ChatGPT/APT画像/AttackAgentDetect
Pinned dataset manifest present: True


## Load verified results

The notebook reads aggregate outputs from `results/`. Run `scripts/run_all.py` before opening it to regenerate the fixed-split experiments and the post-hoc diagnostics. The full 5,000-resample pipeline is intentionally kept in the command-line workflow; this cell verifies the outputs without running the expensive bootstrap twice. Set `REBUILD_RESULTS = True` in the next cell only when a full rebuild is needed. The supplemental feature comparison remains explicitly post-hoc.


In [2]:
REBUILD_RESULTS = False
if REBUILD_RESULTS:
    subprocess.run([sys.executable, "scripts/run_all.py"], check=True, cwd=ROOT)
else:
    print("Using the checked aggregate result files. Set REBUILD_RESULTS = True to regenerate.")

verification = subprocess.run(
    [sys.executable, "scripts/verify_results.py"],
    check=True, cwd=ROOT, capture_output=True, text=True
)
print(verification.stdout)

paths = {
    "audit": ROOT / "results/data-audit.json",
    "primary": ROOT / "results/preliminary.json",
    "sensitivity": ROOT / "results/sensitivity.json",
    "trace": ROOT / "results/trace-baseline.json",
    "split_diagnostics": ROOT / "results/split-diagnostics.json",
}
results = {key: json.loads(path.read_text(encoding="utf-8")) for key, path in paths.items()}
print("Loaded dataset revision:", results["audit"]["revision"])


Using the checked aggregate result files. Set REBUILD_RESULTS = True to regenerate.


{
  "status": "passed",
  "dataset_revision": "178c0da1800f9c07311e1b8d6c61dd8104e7b4ae",
  "human_samples": 47,
  "agent_samples": 194,
  "model_counts": {
    "claude-opus-4.6": 74,
    "claude-sonnet-4.6": 74,
    "gpt-5.5": 46
  },
  "nonempty_matched_tasks": 27,
  "split_sizes": {
    "train": 16,
    "validation": 5,
    "test": 6
  },
  "test_pairs_per_association_method": 351,
  "bootstrap_resamples": 5000,
  "trace_test_balanced_accuracy": 0.9444444444444444,
  "split_diagnostic_repeats": 50,
  "task_group_split_overlap": 0.0
}

Loaded dataset revision: 178c0da1800f9c07311e1b8d6c61dd8104e7b4ae


## Dataset audit and fixed split

In [3]:
import pandas as pd
from IPython.display import display

au = results["audit"]
pr = results["primary"]
data_table = pd.DataFrame([
    ("Human files", au["human_files"]),
    ("Readable human sessions", au["human_samples"]),
    ("Agent/model sessions", au["agent_samples"]),
    ("Shared task IDs before empty-trace filtering", au["matched_task_count"]),
    ("Shared non-empty tasks used", pr["matched_nonempty_tasks"]),
    ("Human sessions on shared task IDs", au["matched_human_samples"]),
    ("Model sessions on shared task IDs", au["matched_agent_samples"]),
    ("Excluded Python tool calls", pr["python_tool_calls_excluded"]),
], columns=["Audit item", "Count"])
display(data_table)

display(pd.DataFrame([
    {"Split": name, "Tasks": row["task_count"], "Human traces": row["human_samples"], "Agent traces": row["agent_samples"]}
    for name, row in pr["splits"].items()
]))

display(pd.DataFrame([
    {"Model group": model, "Sessions": count}
    for model, count in au["agent_samples_by_model"].items()
]))

,Audit item,Count
0,Human files,48
1,Readable human sessions,47
2,Agent/model sessions,194
3,Shared task IDs before empty-trace filtering,31
4,Shared non-empty tasks used,27
5,Human sessions on shared task IDs,45
6,Model sessions on shared task IDs,93
7,Excluded Python tool calls,291


,Split,Tasks,Human traces,Agent traces
0,train,16,23,48
1,validation,5,8,15
2,test,6,9,18


,Model group,Sessions
0,claude-opus-4.6,74
1,claude-sonnet-4.6,74
2,gpt-5.5,46


## Participation classification baselines

All methods use the same six held-out tasks. The simple count-only model is a confound audit, not a deployable detector. Confidence intervals resample tasks, not individual trajectories.

In [4]:
se = results["sensitivity"]["classification"]
tr = results["trace"]
rows = []
method_map = [
    ("Majority", "majority", None),
    ("Log command count", "command_count", None),
    ("Coarse action proportions", "action_mix", None),
    ("Command-name TF-IDF + logistic regression", "command_ngrams", None),
]
for label, key, _ in method_map:
    r = se[key]
    ci = r["task_cluster_bootstrap_95_ci"]["balanced_accuracy"]
    rows.append({
        "Method": label,
        "Balanced accuracy": r["balanced_accuracy"],
        "95% task-bootstrap CI": f"[{ci['lower_95']:.3f}, {ci['upper_95']:.3f}]",
        "Macro-F1": r["macro_f1"],
        "ROC-AUC": r["roc_auc"],
    })
ci = tr["task_cluster_bootstrap_95_ci"]["balanced_accuracy"]
rows.append({
    "Method": "Trace-style TF-IDF + LinearSVC (transfer check)",
    "Balanced accuracy": tr["metrics"]["balanced_accuracy"],
    "95% task-bootstrap CI": f"[{ci['lower_95']:.3f}, {ci['upper_95']:.3f}]",
    "Macro-F1": tr["metrics"]["macro_f1"],
    "ROC-AUC": tr["metrics"]["roc_auc"],
})
classification_table = pd.DataFrame(rows)
display(classification_table.style.format({"Balanced accuracy": "{:.3f}", "Macro-F1": "{:.3f}", "ROC-AUC": "{:.3f}"}))

,Method,Balanced accuracy,95% task-bootstrap CI,Macro-F1,ROC-AUC
0,Majority,0.500,"[0.500, 0.500]",0.400,0.500
1,Log command count,0.583,"[0.496, 0.644]",0.475,0.796
2,Coarse action proportions,0.722,"[0.590, 0.806]",0.665,0.707
3,Command-name TF-IDF + logistic regression,0.889,"[0.778, 1.000]",0.847,1.000
4,Trace-style TF-IDF + LinearSVC (transfer check),0.944,"[0.833, 1.000]",0.921,1.000


## Same-task retrieval by participant pairing

“Same task” is only a benchmark prompt match, not evidence of a common actor, operation, or campaign. The 95% intervals are task-cluster bootstrap diagnostics over six test tasks.

In [5]:
retrieval = results["sensitivity"]["same_task_retrieval"]
retrieval_rows = []
for method, label in [("command_cosine", "Command-name cosine"), ("procgrep_jsd_similarity", "ProcGrep procedure distribution")]:
    for pair in ("human-human", "agent-human", "agent-agent"):
        r = retrieval[method][pair]
        ci = r["task_cluster_bootstrap_95_ci"]
        retrieval_rows.append({
            "Representation": label,
            "Pair": pair.replace("-", "–"),
            "AUC": r["pair_auc"],
            "95% task-bootstrap CI": f"[{ci['lower_95']:.3f}, {ci['upper_95']:.3f}]",
            "Positive tasks": r["independent_positive_tasks"],
            "Positive pairs": r["same_task_pairs"],
            "Negative task-pairs": r["independent_negative_task_pairs"],
        })
retrieval_table = pd.DataFrame(retrieval_rows)
display(retrieval_table.style.format({"AUC": "{:.3f}"}))

,Representation,Pair,AUC,95% task-bootstrap CI,Positive tasks,Positive pairs,Negative task-pairs
0,Command-name cosine,human–human,0.621,"[0.438, 0.875]",3,3,15
1,Command-name cosine,agent–human,0.511,"[0.396, 0.667]",6,27,15
2,Command-name cosine,agent–agent,0.581,"[0.370, 0.825]",6,18,15
3,ProcGrep procedure distribution,human–human,0.879,"[0.643, 1.000]",3,3,15
4,ProcGrep procedure distribution,agent–human,0.533,"[0.446, 0.676]",6,27,15
5,ProcGrep procedure distribution,agent–agent,0.475,"[0.398, 0.586]",6,18,15


## Does participation-aware thresholding help the association proxy?

In [6]:
downstream_rows = []
for method, result in pr["association"].items():
    for policy, row in result["downstream_test"].items():
        downstream_rows.append({
            "Representation": method,
            "Policy": policy,
            "Balanced accuracy": row["balanced_accuracy"],
            "False split rate": row["false_split_rate"],
            "False merge rate": row["false_merge_rate"],
            "Pairs": row["pairs"],
            "Same-task pairs": row["same_task_pairs"],
        })
display(pd.DataFrame(downstream_rows).style.format({
    "Balanced accuracy": "{:.3f}", "False split rate": "{:.3f}", "False merge rate": "{:.3f}"
}))

,Representation,Policy,Balanced accuracy,False split rate,False merge rate,Pairs,Same-task pairs
0,command_cosine,common,0.508,0.333,0.650,351,48
1,command_cosine,uniform_half,0.490,1.000,0.020,351,48
2,command_cosine,oracle_groups,0.528,0.354,0.591,351,48
3,command_cosine,predicted_groups,0.518,0.479,0.485,351,48
4,procgrep_jsd_similarity,common,0.507,0.771,0.215,351,48
5,procgrep_jsd_similarity,uniform_half,0.521,0.833,0.125,351,48
6,procgrep_jsd_similarity,oracle_groups,0.525,0.792,0.158,351,48
7,procgrep_jsd_similarity,predicted_groups,0.519,0.833,0.129,351,48


## Test-set case counts without exposing command content

In [7]:
sys.path.insert(0, str(ROOT))
from attackagent.traces import load_observations
from scripts.run_exploration import split_tasks

observations = [row for row in load_observations() if row.names]
test_tasks = split_tasks(observations)["test"]
case_rows = []
for task in sorted(test_tasks):
    group = [row for row in observations if row.task == task]
    human = [len(row.names) for row in group if row.label == "human"]
    agent = [len(row.names) for row in group if row.label == "agent"]
    case_rows.append({
        "Held-out task": task.rsplit("/", 1)[-1],
        "Human sessions": len(human),
        "Agent sessions": len(agent),
        "Median recognized command heads, human": float(pd.Series(human).median()) if human else None,
        "Median recognized command heads, agent": float(pd.Series(agent).median()) if agent else None,
    })
display(pd.DataFrame(case_rows))

,Held-out task,Human sessions,Agent sessions,"Median recognized command heads, human","Median recognized command heads, agent"
0,task_11,1,3,7.0,7.0
1,task_36,1,3,2.0,4.0
2,task_4,1,3,3.0,4.0
3,task_13,2,3,3.0,3.0
4,task_45,2,3,2.0,5.0
5,task_83,2,3,2.0,5.0


## Visual diagnostic

In [8]:
import matplotlib.pyplot as plt
import numpy as np

PALETTE = {
    "orange": "#FEBC78",
    "mint": "#94D5B2",
    "green": "#74BF96",
    "pale_blue": "#99CDFC",
    "blue": "#45A6FC",
    "ink": "#263746",
    "muted": "#617384",
    "line": "#D8E3EC",
}
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.edgecolor": PALETTE["line"],
    "axes.labelcolor": PALETTE["ink"],
    "xtick.color": PALETTE["muted"],
    "ytick.color": PALETTE["muted"],
    "text.color": PALETTE["ink"],
    "svg.fonttype": "none",
})
labels = [row["Method"].replace("Command-name ", "Command ").replace(" (transfer check)", "") for row in rows]
values = [row["Balanced accuracy"] for row in rows]
ci_pairs = []
for key in ("majority", "command_count", "action_mix", "command_ngrams"):
    interval = se[key]["task_cluster_bootstrap_95_ci"]["balanced_accuracy"]
    ci_pairs.append((interval["lower_95"], interval["upper_95"]))
interval = tr["task_cluster_bootstrap_95_ci"]["balanced_accuracy"]
ci_pairs.append((interval["lower_95"], interval["upper_95"]))
errors = np.array([[v - lo for v, (lo, hi) in zip(values, ci_pairs)], [hi - v for v, (lo, hi) in zip(values, ci_pairs)]])
bar_colors = [PALETTE["muted"], PALETTE["orange"], PALETTE["mint"], PALETTE["blue"], PALETTE["pale_blue"]]
y = np.arange(len(labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={"width_ratios": [1.05, 1.2]})
ax1.barh(y, values, color=bar_colors, height=.62, edgecolor="white", linewidth=1)
ax1.errorbar(values, y, xerr=errors, fmt="none", ecolor=PALETTE["ink"], capsize=3, linewidth=1)
ax1.axvline(.5, color=PALETTE["muted"], linestyle=(0, (3, 3)), linewidth=1)
ax1.set_yticks(y, labels)
ax1.invert_yaxis()
ax1.set_xlim(0.35, 1.02)
ax1.set_xlabel("Balanced accuracy (95% task-bootstrap interval)")
ax1.set_title("Participation-label classification", loc="left", fontweight="bold", color=PALETTE["ink"])
ax1.grid(axis="x", color=PALETTE["line"], linewidth=.7)
ax1.set_axisbelow(True)
for spine in ("top", "right", "left"):
    ax1.spines[spine].set_visible(False)

pairs = ["human-human", "agent-human", "agent-agent"]
pretty_pairs = ["Human–human", "Human–Agent", "Agent–Agent"]
methods = [("command_cosine", "Command cosine", PALETTE["blue"]), ("procgrep_jsd_similarity", "ProcGrep", PALETTE["green"])]
for offset, (method, label, color) in zip([-.12, .12], methods):
    vals, lows, highs = [], [], []
    for pair in pairs:
        item = retrieval[method][pair]
        ci = item["task_cluster_bootstrap_95_ci"]
        vals.append(item["pair_auc"])
        lows.append(ci["lower_95"])
        highs.append(ci["upper_95"])
    ypos = np.arange(3) + offset
    ax2.errorbar(vals, ypos, xerr=[np.array(vals)-np.array(lows), np.array(highs)-np.array(vals)], fmt="o", color=color, ecolor=color, capsize=3, linewidth=1.7, markersize=6, label=label)
ax2.axvline(.5, color=PALETTE["muted"], linestyle=(0, (3, 3)), linewidth=1)
ax2.set_yticks(np.arange(3), pretty_pairs)
ax2.invert_yaxis()
ax2.set_xlim(.25, 1.02)
ax2.set_xlabel("Same-task retrieval AUC (95% task-bootstrap interval)")
ax2.set_title("Same-task retrieval by pair type", loc="left", fontweight="bold", color=PALETTE["ink"])
ax2.grid(axis="x", color=PALETTE["line"], linewidth=.7)
ax2.set_axisbelow(True)
for spine in ("top", "right", "left"):
    ax2.spines[spine].set_visible(False)
handles, legend_labels = ax2.get_legend_handles_labels()
fig.legend(handles, legend_labels, frameon=False, loc="upper right", bbox_to_anchor=(.98, .96), ncol=2)
fig.suptitle("Classification strength does not imply association signal", x=.04, y=.99, ha="left", fontsize=15, fontweight="bold", color=PALETTE["ink"])
fig.text(.04, .01, "Exploratory benchmark proxy · 27 held-out traces · 6 independent tasks · same task ≠ same attacker or campaign", color=PALETTE["muted"], fontsize=9)
fig.tight_layout(rect=[0, .055, 1, .87])
figure_path = ROOT / "site/association-results.svg"
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, bbox_inches="tight", facecolor="white")
plt.close(fig)

mobile, (m1, m2) = plt.subplots(2, 1, figsize=(7.2, 8.8), gridspec_kw={"height_ratios": [1.05, 1]})
short_labels = ["Majority", "Command count", "Action mix", "TF-IDF + LR", "Trace-style + SVM"]
m1.barh(y, values, color=bar_colors, height=.62, edgecolor="white", linewidth=1)
m1.errorbar(values, y, xerr=errors, fmt="none", ecolor=PALETTE["ink"], capsize=3, linewidth=1)
m1.axvline(.5, color=PALETTE["muted"], linestyle=(0, (3, 3)), linewidth=1)
m1.set_yticks(y, short_labels)
m1.tick_params(axis="y", labelsize=9)
m1.invert_yaxis()
m1.set_xlim(0.35, 1.02)
m1.set_xlabel("Balanced accuracy (95% task-bootstrap interval)")
m1.set_title("Participation-label classification", loc="left", fontweight="bold", color=PALETTE["ink"])
m1.grid(axis="x", color=PALETTE["line"], linewidth=.7)
m1.set_axisbelow(True)
for spine in ("top", "right", "left"):
    m1.spines[spine].set_visible(False)

for offset, (method, label, color) in zip([-.12, .12], methods):
    vals, lows, highs = [], [], []
    for pair in pairs:
        item = retrieval[method][pair]
        ci = item["task_cluster_bootstrap_95_ci"]
        vals.append(item["pair_auc"])
        lows.append(ci["lower_95"])
        highs.append(ci["upper_95"])
    ypos = np.arange(3) + offset
    m2.errorbar(vals, ypos, xerr=[np.array(vals)-np.array(lows), np.array(highs)-np.array(vals)], fmt="o", color=color, ecolor=color, capsize=3, linewidth=1.7, markersize=6, label=label)
m2.axvline(.5, color=PALETTE["muted"], linestyle=(0, (3, 3)), linewidth=1)
m2.set_yticks(np.arange(3), ["Human / human", "Human / Agent", "Agent / Agent"])
m2.tick_params(axis="y", labelsize=9)
m2.invert_yaxis()
m2.set_xlim(.25, 1.02)
m2.set_xlabel("Same-task retrieval AUC (95% task-bootstrap interval)")
m2.set_title("Same-task retrieval (not campaign linkage)", loc="left", fontweight="bold", color=PALETTE["ink"])
m2.grid(axis="x", color=PALETTE["line"], linewidth=.7)
m2.set_axisbelow(True)
for spine in ("top", "right", "left"):
    m2.spines[spine].set_visible(False)
mobile_handles, mobile_labels = m2.get_legend_handles_labels()
mobile.legend(mobile_handles, mobile_labels, frameon=False, loc="upper right", bbox_to_anchor=(.97, .96), ncol=2)
mobile.suptitle("High classification score ≠ association evidence", x=.05, y=.99, ha="left", fontsize=15, fontweight="bold", color=PALETTE["ink"])
mobile.text(.05, .01, "Exploratory proxy · 6 held-out tasks · same task ≠ same activity", color=PALETTE["muted"], fontsize=9)
mobile.tight_layout(rect=[0, .055, 1, .88], h_pad=2.4)
mobile_path = ROOT / "site/association-results-mobile.svg"
mobile.savefig(mobile_path, bbox_inches="tight", facecolor="white")
plt.close(mobile)
for rendered_svg in (figure_path, mobile_path):
    lines = rendered_svg.read_text(encoding="utf-8").splitlines()
    rendered_svg.write_text("\n".join(line.rstrip() for line in lines) + "\n", encoding="utf-8")
plt.show()
print("Saved vector figures:", figure_path.relative_to(ROOT), "and", mobile_path.relative_to(ROOT))

Saved vector figures: site/association-results.svg and site/association-results-mobile.svg


## Split sensitivity: session-random vs. task-held-out

This post-hoc diagnostic repeats the same fixed TF-IDF/logistic baseline over 50 seeds. The random-session split may place the same benchmark task in train and test; the grouped split holds out complete tasks. Repeats share records, so spread is descriptive and is not a confidence interval. The class label remains confounded with the logging pipeline.

In [9]:
sd = results["split_diagnostics"]
split_rows = []
for label, key in [("Random session split", "random_session_split"), ("Task-group split", "task_group_split")]:
    entry = sd[key]
    split_rows.append({
        "Evaluation split": label,
        "Mean BAcc": entry["metrics"]["balanced_accuracy"]["mean"],
        "Across-split SD": entry["metrics"]["balanced_accuracy"]["sd_across_splits_descriptive_only"],
        "Mean train/test task overlap": entry["mean_train_test_task_overlap"],
    })
display(pd.DataFrame(split_rows).round(3))
print("These repeated splits are sensitivity checks, not independent replicates or confidence intervals.")

,Evaluation split,Mean BAcc,Across-split SD,Mean train/test task overlap
0,Random session split,0.864,0.070,17.68
1,Task-group split,0.864,0.077,0.00


These repeated splits are sensitivity checks, not independent replicates or confidence intervals.


## Interpretation

The main diagnostic is a mismatch: held-out classification is strong, but cross-source human–Agent same-task retrieval is close to chance. The classification label is also confounded with logging modality, and command-count/action-mix baselines recover some class signal. Group-specific association thresholds do not materially improve the task proxy.

This is evidence that the current public benchmark cannot establish the proposed causal story. It is not evidence that real APT activity association fails. A decisive study needs one external telemetry pipeline, known human/script/Agent control conditions, and independent same/different activity labels.

## References

- [OHAI framework, CMU SEI](https://www.sei.cmu.edu/library/observational-humanai-ohai-a-defender-attribution-framework-for-distinguishing-human-vs-ai-threats/)
- [Trace: model-family fingerprinting of attack Agents](https://arxiv.org/abs/2605.01186)
- [LIDAR: active fingerprinting through agentic behavior](https://arxiv.org/abs/2609.28559)
- [AgentTrace: execution-level model fingerprinting](https://arxiv.org/abs/2610.32092)
- [ProcGrep: procedural representations of Agent trajectories](https://arxiv.org/abs/2606.16988)
- [ATLAS: sequence-based attack investigation](https://www.usenix.org/conference/usenixsecurity21/presentation/alsaheel)
- [POIROT: CTI-to-provenance graph alignment](https://arxiv.org/abs/1910.00056)
- [OpTC data release and red-team ground truth](https://github.com/FiveDirections/OpTC-data)
- [SAGA synthetic audit logs](https://www.saga-cyber.net/dataset)
- [Offensive Cybersecurity Time Horizons dataset](https://lyptusresearch.org/research/offensive-cyber-time-horizons)